In [2]:
using Random
using Statistics
using Distributions
using ShiftedArrays
using CategoricalArrays
# using StatsPlots
using HypothesisTests

using DataFrames
using DataFramesMeta
using SpecialFunctions
using LinearAlgebra
using StatsFuns
using MixedModels
using CodecZlib
using JLD2
using Serialization
using LaTeXStrings
using ShiftedArrays: lag
using Latexify
using Markdown
using HypothesisTests
using Printf

In [3]:
base_dir = ".."
include("$(base_dir)/code/regression_sim.jl")

add_all_regressors! (generic function with 1 method)

In [4]:
# Per-subject BLUPs for the MB (rewardₜ₋₁ & lag1_neighborboat_reg) and SR
# (rewardₜ₋₁ & lag1_policy_reg) interaction slopes (random slopes in the full-model
# RE term). Same extraction pattern as save_nb_blups!.
function mb_sr_blups(model)
    mb = "rewardₜ₋₁ & lag1_neighborboat_reg"
    sr = "rewardₜ₋₁ & lag1_policy_reg"
    fn = coefnames(model); re = only(ranef(model)); rn = only(model.reterms).cnames
    mb_s = fixef(model)[findfirst(==(mb), fn)] .+ re[findfirst(==(mb), rn), :]
    sr_s = fixef(model)[findfirst(==(sr), fn)] .+ re[findfirst(==(sr), rn), :]
    (mb_s, sr_s)
end

# Correct MB-vs-SR test: per-subject difference carries the coefficient covariance
# (unlike the old ind_test which assumed independent SEs).
function mb_sr_blup_test(model, label)
    mb_s, sr_s = mb_sr_blups(model)
    d  = mb_s .- sr_s
    tt = OneSampleTTest(d); wt = SignedRankTest(d)
    @printf("%-14s n=%3d  MB=%+.3f  SR=%+.3f  MB-SR=%+.3f  t=%6.2f  p=%.2e  (Wilcoxon p=%.2e)\n",
            label, length(d), mean(mb_s), mean(sr_s), mean(d), tt.t, pvalue(tt), pvalue(wt))
    (; label, mb_s, sr_s, d, t = tt.t, p = pvalue(tt), p_wilcoxon = pvalue(wt))
end

# Wald test of the fixed-effect contrast β_MB − β_SR.
function mb_sr_wald_test(model, label)
    fn = coefnames(model)
    β  = coef(model)
    V  = vcov(model)
    i_mb = findfirst(==("rewardₜ₋₁ & lag1_neighborboat_reg"), fn)
    i_sr = findfirst(==("rewardₜ₋₁ & lag1_policy_reg"),        fn)

    se_mb = sqrt(V[i_mb, i_mb])                                   # SE of β_MB
    se_sr = sqrt(V[i_sr, i_sr])                                   # SE of β_SR
    est   = β[i_mb] - β[i_sr]
    se    = sqrt(V[i_mb,i_mb] + V[i_sr,i_sr] - 2*V[i_mb,i_sr])    # SE of the contrast
    z     = est / se
    p     = 2*ccdf(Normal(), abs(z))

    @printf("%-14s  MB=%+.3f (se=%.3f)  SR=%+.3f (se=%.3f)  MB-SR=%+.3f (se=%.3f)  z=%6.2f  p=%.2e\n",
            label, β[i_mb], se_mb, β[i_sr], se_sr, est, se, z, p)
    (; label, mb=β[i_mb], se_mb, sr=β[i_sr], se_sr, est, se, z, p)
end


mb_sr_wald_test (generic function with 1 method)

In [5]:
# Load pre-saved MixedModel objects (avoids re-running simulations)
m_sr   = load("$(base_dir)/results/m_sr.jld2",     "m_sr")
m_mb   = load("$(base_dir)/results/m_mb.jld2",     "m_mb")
m_lrl  = load("$(base_dir)/results/m_lrl.jld2",    "m_lrl")
m_mbsr = load("$(base_dir)/results/m_hybrid_2.jld2", "m_hybrid")
m_sris = load("$(base_dir)/results/m_sris.jld2",   "m_sris")
m_part = load("$(base_dir)/results/m_part.jld2",   "m_part")

|                                   |    Est. |     SE |      z |      p | σ_subject |
|:--------------------------------- | -------:| ------:| ------:| ------:| ---------:|
| (Intercept)                       |  0.2080 | 0.0520 |   4.00 | <1e-04 |    0.2625 |
| rewardₜ₋₁                         |  0.0020 | 0.0921 |   0.02 | 0.9828 |    0.2034 |
| lag1_neighborboat_reg             | -0.6824 | 0.0523 | -13.04 | <1e-38 |    0.3327 |
| lag2_neighborboat_reg             | -0.1317 | 0.0407 |  -3.24 | 0.0012 |           |
| lag3_neighborboat_reg             | -0.1166 | 0.0402 |  -2.90 | 0.0037 |           |
| lag4_neighborboat_reg             | -0.0372 | 0.0400 |  -0.93 | 0.3533 |           |
| lag5_neighborboat_reg             | -0.1174 | 0.0397 |  -2.96 | 0.0031 |           |
| lag1_sameboat_reg                 |  0.4745 | 0.0404 |  11.76 | <1e-31 |           |
| lag2_sameboat_reg                 |  0.1678 | 0.0401 |   4.19 | <1e-04 |           |
| lag3_sameboat_reg                 |  0.0544 | 0.0402 |   1.35 | 0.1764 |           |
| lag4_sameboat_reg                 |  0.0948 | 0.0401 |   2.36 | 0.0181 |           |
| lag5_sameboat_reg                 |  0.0277 | 0.0396 |   0.70 | 0.4840 |           |
| lag1_policy_reg                   | -0.0464 | 0.0445 |  -1.04 | 0.2978 |    0.1371 |
| lag2_policy_reg                   | -0.0525 | 0.0444 |  -1.18 | 0.2366 |           |
| lag3_policy_reg                   |  0.0375 | 0.0457 |   0.82 | 0.4123 |           |
| lag4_policy_reg                   |  0.0043 | 0.0457 |   0.09 | 0.9254 |           |
| lag5_policy_reg                   |  0.0512 | 0.0437 |   1.17 | 0.2404 |           |
| lag1_oppislandavgboat_reg         | -1.5582 | 0.0614 | -25.38 | <1e-99 |           |
| lag2_oppislandavgboat_reg         | -0.3220 | 0.0598 |  -5.38 | <1e-07 |           |
| lag3_oppislandavgboat_reg         | -0.1305 | 0.0583 |  -2.24 | 0.0253 |           |
| lag4_oppislandavgboat_reg         | -0.1290 | 0.0579 |  -2.23 | 0.0261 |           |
| lag5_oppislandavgboat_reg         | -0.0994 | 0.0575 |  -1.73 | 0.0839 |           |
| lag1_choice_autoreg               |  1.2875 | 0.0400 |  32.19 | <1e-99 |           |
| lag2_choice_autoreg               |  0.9544 | 0.0416 |  22.95 | <1e-99 |           |
| lag3_choice_autoreg               |  0.0557 | 0.0431 |   1.29 | 0.1959 |           |
| lag4_choice_autoreg               |  0.4887 | 0.0422 |  11.58 | <1e-30 |           |
| lag5_choice_autoreg               |  0.0564 | 0.0412 |   1.37 | 0.1709 |           |
| rewardₜ₋₁ & lag1_neighborboat_reg |  0.7411 | 0.1081 |   6.86 | <1e-11 |    0.7229 |
| rewardₜ₋₁ & lag2_neighborboat_reg |  0.0940 | 0.0807 |   1.16 | 0.2441 |           |
| rewardₜ₋₁ & lag3_neighborboat_reg |  0.0195 | 0.0799 |   0.24 | 0.8076 |           |
| rewardₜ₋₁ & lag4_neighborboat_reg | -0.0755 | 0.0795 |  -0.95 | 0.3420 |           |
| rewardₜ₋₁ & lag5_neighborboat_reg | -0.0569 | 0.0791 |  -0.72 | 0.4723 |           |
| rewardₜ₋₁ & lag1_sameboat_reg     |  0.1833 | 0.0803 |   2.28 | 0.0224 |           |
| rewardₜ₋₁ & lag2_sameboat_reg     | -0.0106 | 0.0797 |  -0.13 | 0.8941 |           |
| rewardₜ₋₁ & lag3_sameboat_reg     | -0.0451 | 0.0800 |  -0.56 | 0.5730 |           |
| rewardₜ₋₁ & lag4_sameboat_reg     |  0.0378 | 0.0798 |   0.47 | 0.6361 |           |
| rewardₜ₋₁ & lag5_sameboat_reg     |  0.0195 | 0.0788 |   0.25 | 0.8049 |           |
| rewardₜ₋₁ & lag1_policy_reg       |  0.3883 | 0.0934 |   4.16 | <1e-04 |    0.3859 |
| rewardₜ₋₁ & lag2_policy_reg       |  0.1061 | 0.0890 |   1.19 | 0.2330 |           |
| rewardₜ₋₁ & lag3_policy_reg       |  0.2611 | 0.0916 |   2.85 | 0.0044 |           |
| rewardₜ₋₁ & lag4_policy_reg       | -0.1311 | 0.0915 |  -1.43 | 0.1517 |           |
| rewardₜ₋₁ & lag5_policy_reg       | -0.1530 | 0.0874 |  -1.75 | 0.0801 |           |


In [6]:
println("=== Wald contrast: MB:rwd vs SR:rwd (positive = MB > SR) ===")
mb_sr_wald_test(m_part, "Participants")
mb_sr_wald_test(m_mb,   "MB")
mb_sr_wald_test(m_sr,   "SR")
mb_sr_wald_test(m_lrl,  "Linear RL")
mb_sr_wald_test(m_mbsr, "Hybrid SR+MB")
mb_sr_wald_test(m_sris, "SR-IS")

=== Wald contrast: MB:rwd vs SR:rwd (positive = MB > SR) ===
Participants    MB=+0.741 (se=0.108)  SR=+0.388 (se=0.093)  MB-SR=+0.353 (se=0.142)  z=  2.48  p=1.32e-02
MB              MB=+0.906 (se=0.066)  SR=-0.011 (se=0.069)  MB-SR=+0.917 (se=0.091)  z= 10.05  p=9.32e-24
SR              MB=+0.030 (se=0.056)  SR=+0.611 (se=0.077)  MB-SR=-0.582 (se=0.089)  z= -6.52  p=7.07e-11
Linear RL       MB=+1.423 (se=0.069)  SR=-0.252 (se=0.070)  MB-SR=+1.675 (se=0.103)  z= 16.26  p=1.98e-59
Hybrid SR+MB    MB=+0.357 (se=0.068)  SR=+0.648 (se=0.074)  MB-SR=-0.292 (se=0.098)  z= -2.99  p=2.78e-03
SR-IS           MB=+0.379 (se=0.053)  SR=+0.178 (se=0.053)  MB-SR=+0.200 (se=0.069)  z=  2.88  p=3.95e-03


(label = "SR-IS", mb = 0.3786690796376538, se_mb = 0.052882377458966165, sr = 0.17847831565982217, se_sr = 0.05255574392628225, est = 0.20019076397783162, se = 0.06945052544401245, z = 2.8824945916242983, p = 0.003945399088792375)

In [7]:
# println("=== Per-subject BLUP paired test: MB:rwd vs SR:rwd (positive = MB > SR) ===")
# res = Dict(
#     "participants" => mb_sr_blup_test(m_part, "Participants"),
#     "mb"           => mb_sr_blup_test(m_mb,   "MB"),
#     "sr"           => mb_sr_blup_test(m_sr,   "SR"),
#     "lrl"          => mb_sr_blup_test(m_lrl,  "Linear RL"),
#     "mb_sr"        => mb_sr_blup_test(m_mbsr, "Hybrid SR+MB"),
#     "sr_is"        => mb_sr_blup_test(m_sris, "SR-IS"),
# )

In [8]:
# save("/Users/abizzle/Research/SR-IS/src/kahn-analysis/results/m_part.jld2", "m_part", m_part)